# METODE 5: THE PROPOSED MODEL (FULL ARCHITECTURE)
**Arsitektur:** Fusi CQT-LFCC + AASIST Backbone + Temporal Attention Mechanism  
**Peran:** Sebagai model utama (*Proposed Method*) yang diharapkan menghasilkan nilai EER dan min t-DCF terendah. Model ini membuktikan bahwa fusi fitur hibrida yang dipadukan dengan pemantauan dependensi waktu (*temporal attention*) adalah solusi terbaik untuk mendeteksi *deepfake* modern.

Notebook ini diorganisasikan ke dalam beberapa bagian berikut:
1. **Determinisme & Pengaturan Random Seed:** Mengatur seed acak agar hasil eksperimen dapat diulang secara deterministik.
2. **Persiapan Environment (Clone Repo AASIST):** Mengunduh kode AASIST resmi dan mengatur path agar modul dapat diimpor.
3. **Front-End (Ekstraksi & Fusi Fitur CQT-LFCC):** Pengekstrakan CQT dan LFCC, interpolasi penyamaan dimensi waktu, dan konkatenasi frekuensi.
4. **Mekanisme Atensi Temporal (Temporal Attention):** Penjelasan matematika mekanisme atensi berbasis *Query, Key, Value* untuk menangkap dependensi sekuensial jangka panjang.
5. **Proposed Model (AASIST + Adapter + Temporal Attention):** Integrasi penuh Adapter Layer, backbone graf AASIST, dan modul Temporal Attention.
6. **Tempat Import Dataset & Dataloader (ASVspoof 2019 LA):** Pemuatan dataset lokal berupa sinyal audio mentah.
7. **Cross-Dataset Evaluation (ASVspoof 5):** Memuat dataset streaming ASVspoof 5 dari Hugging Face dan mengimplementasikan *Stratified Sampling* untuk mengambil subset berimbang (maksimal 5.000 sampel).
8. **Metrik Evaluasi:** Perhitungan EER dan min t-DCF.
9. **Training & Validation Loop:** Loop pelatihan penuh menggunakan AdamW optimizer, CosineAnnealingLR, dan loss function.

## 1. Determinisme & Pengaturan Random Seed
Agar hasil eksperimen bersifat deterministik, kita mengunci seed acak pada Python, NumPy, dan PyTorch.

In [ ]:
import os
import random
import numpy as np
import torch

def set_seed(seed=1234):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(1234)
print("Determinisme aktif. Random seed diatur ke 1234.")

## 2. Persiapan Environment (Clone Repo AASIST)
Untuk menggunakan arsitektur AASIST resmi, kita mengunduh repositori Clova AI AASIST langsung ke dalam *working directory* Kaggle atau lokal. Kita juga menambahkan direktori repositori tersebut ke dalam `sys.path` agar modul model dapat diimpor secara langsung.

In [ ]:
import sys
import subprocess

# Mengunduh repositori AASIST resmi jika belum diunduh
if not os.path.exists('./aasist'):
    print("Mengunduh repositori AASIST dari Clova AI...")
    subprocess.run(["git", "clone", "https://github.com/clovaai/aasist.git"])
else:
    print("Repositori AASIST sudah tersedia.")

# Menambahkan ke sys.path
sys.path.append('./aasist')
print("Path './aasist' berhasil ditambahkan ke sys.path.")

## 3. Front-End (Ekstraksi & Fusi Fitur CQT-LFCC)
Modul front-end mengekstrak fitur CQT dan LFCC dari audio waveform mentah 16kHz, menyelaraskan panjang waktu (*time frames*) menggunakan interpolasi bilinear `torch.nn.functional.interpolate` ke panjang target (`target_time_len = 2000`), lalu menggabungkannya pada dimensi frekuensi.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import librosa

class CQTLFCCFeatureExtractor(nn.Module):
    """
    Ekstraktor Fitur Hibrida CQT-LFCC dengan penyamaan dimensi waktu melalui interpolasi.
    """
    def __init__(self, sample_rate=16000, n_bins_cqt=80, n_lfcc=20, target_time_len=2000):
        super(CQTLFCCFeatureExtractor, self).__init__()
        self.sample_rate = sample_rate
        self.n_bins_cqt = n_bins_cqt
        self.n_lfcc = n_lfcc
        self.target_time_len = target_time_len
        
        # Transformasi LFCC bawaan torchaudio
        self.lfcc_transform = T.LFCC(
            sample_rate=sample_rate,
            n_filter=40,
            n_lfcc=n_lfcc,
            speckwargs={
                "n_fft": 512,
                "win_length": 400,
                "hop_length": 160,
                "center": True
            }
        )

    def forward(self, waveform):
        # Input waveform shape: (batch_size, num_samples)
        batch_size = waveform.shape[0]
        
        # 1. Ekstraksi CQT per sampel menggunakan librosa
        cqt_list = []
        for i in range(batch_size):
            y = waveform[i].cpu().numpy()
            cqt = librosa.cqt(y, sr=self.sample_rate, hop_length=256, n_bins=self.n_bins_cqt)
            cqt_db = librosa.amplitude_to_db(np.abs(cqt), ref=np.max)
            cqt_list.append(torch.tensor(cqt_db, dtype=torch.float32).to(waveform.device))
            
        cqt_feat = torch.stack(cqt_list, dim=0) # Shape: (batch_size, n_bins_cqt, t_cqt)
        
        # 2. Ekstraksi LFCC menggunakan torchaudio
        lfcc_feat = self.lfcc_transform(waveform) # Shape: (batch_size, n_lfcc, t_lfcc)
        
        # 3. Interpolasi bilinear untuk menyamakan dimensi waktu
        # CQT resize
        cqt_interp = F.interpolate(
            cqt_feat.unsqueeze(1), 
            size=(self.n_bins_cqt, self.target_time_len), 
            mode='bilinear', 
            align_corners=False
        ).squeeze(1)
        
        # LFCC resize
        lfcc_interp = F.interpolate(
            lfcc_feat.unsqueeze(1), 
            size=(self.n_lfcc, self.target_time_len), 
            mode='bilinear', 
            align_corners=False
        ).squeeze(1)
        
        # 4. Konkatenasi pada dimensi frekuensi (n_bins_cqt + n_lfcc)
        fused_feat = torch.cat([cqt_interp, lfcc_interp], dim=1) # Shape: (batch, n_bins_cqt + n_lfcc, target_time_len)
        
        # Menambahkan dimensi channel
        return fused_feat.unsqueeze(1) # Shape: (batch_size, 1, n_bins_cqt + n_lfcc, target_time_len)

print("Modul ekstraksi & fusi CQT-LFCC berhasil didefinisikan.")

## 4. Mekanisme Atensi Temporal (Temporal Attention Mechanism)
Untuk memfokuskan model pada bingkai waktu (*time frames*) tertentu yang mengandung artifak *deepfake* (seperti jeda bicara tidak wajar atau transisi fase yang ganjil), kita mengganti *readout pooling* temporal bawaan dengan mekanisme **Temporal Attention**.

Secara matematis, mekanisme atensi berbasis *Query, Key, Value* ini diformulasikan sebagai:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

Di mana:
- **Key ($K$) & Value ($V$):** Representasi temporal keluaran dari Graph Attention Network, $H_T \in \mathbb{R}^{B \times N_T \times D}$.
- **Query ($Q$):** Vektor parameter yang dapat dipelajari (*learnable query vector*), $Q_{param} \in \mathbb{R}^{1 \times 1 \times D}$, yang merepresentasikan pola anomali artifak *deepfake* global.
- **$d_k$:** Dimensi representasi tersembunyi ($D$) untuk menstabilkan gradien saat perkalian dot product.

Mekanisme ini memungkinkan model Proposed memusatkan perhatian (*saliency*) secara adaptif ke rentang waktu yang mencurigakan, menjaga informasi sekuensial jangka panjang (*long-term sequential dependencies*), alih-alih merata-ratakan atau mengambil nilai maksimum dari seluruh bingkai waktu.

In [ ]:
class TemporalAttention(nn.Module):
    """
    Mekanisme Atensi Temporal berbasis Multi-head Attention untuk pooling sekuens temporal.
    """
    def __init__(self, embed_dim=32, num_heads=2):
        super(TemporalAttention, self).__init__()
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        # Learnable Query Parameter
        self.query_param = nn.Parameter(torch.randn(1, 1, embed_dim))

    def forward(self, x):
        # Input x (Key & Value) shape: (batch_size, seq_len, embed_dim)
        batch_size = x.size(0)
        
        # Memperluas Query ke ukuran batch
        query = self.query_param.expand(batch_size, -1, -1) # Shape: (batch_size, 1, embed_dim)
        
        # Menghitung output atensi
        # attn_output shape: (batch_size, 1, embed_dim)
        attn_output, _ = self.mha(query, x, x)
        
        # Menghilangkan dimensi waktu yang ter-pool
        return attn_output.squeeze(1) # Shape: (batch_size, embed_dim)

print("Modul Temporal Attention berhasil dibuat.")

## 5. Proposed Model (AASIST + Adapter + Temporal Attention)
Di bawah ini adalah wrapper Proposed Model yang menyatukan seluruh komponen:
1. **Adapter Layer:** Conv2D + Adaptive Pooling untuk menyesuaikan input fusi CQT-LFCC ke backbone AASIST.
2. **AASIST Graph Extraction:** Mengekstrak representasi graf spasio-temporal.
3. **Temporal Attention Pooling:** Menggantikan pooling temporal rata-rata/maksimum standar dengan modul atensi sekuensial temporal kustom.
4. **Classifier Head:** Linear Classifier 2 kelas.

In [ ]:
class Proposed_AASIST_Model(nn.Module):
    """
    Proposed Model (Metode 5) menggabungkan Fusi CQT-LFCC, Adapter, Backbone AASIST, 
    dan Temporal Attention Classifier Head.
    """
    def __init__(self, aasist_model, fusion_dim=100, embed_dim=32):
        super(Proposed_AASIST_Model, self).__init__()
        self.aasist = aasist_model
        
        # Adapter layer untuk memproyeksikan dimensi frekuensi dari 100 ke 23
        self.adapter = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(3, 3), padding=(1, 1)),
            nn.SELU(inplace=True),
            nn.BatchNorm2d(1),
            nn.AdaptiveAvgPool2d((23, None))
        )
        
        # Modul Temporal Attention untuk pooling representasi waktu
        self.temporal_attention = TemporalAttention(embed_dim=embed_dim, num_heads=2)
        
        # Redefinisi classifier head: temporal_attn(32) + S_max(32) + S_avg(32) + master(32) = 128
        self.out_layer = nn.Linear(4 * embed_dim, 2)

    def forward(self, x):
        # Input x shape: (batch_size, 1, fusion_dim, target_time_len)
        
        # 1. Proyeksi Adapter
        x = self.adapter(x) # Output shape: (batch_size, 1, 23, target_time_len)
        
        # 2. Forward pass melewati Encoder AASIST
        e = self.aasist.encoder(x)
        
        # 3. Spectral Graph Attention Network (GAT-S)
        e_S, _ = torch.max(torch.abs(e), dim=3) # Max pooling sepanjang waktu
        e_S = e_S.transpose(1, 2) + self.aasist.pos_S
        
        gat_S = self.aasist.GAT_layer_S(e_S)
        out_S = self.aasist.pool_S(gat_S)
        
        # 4. Temporal Graph Attention Network (GAT-T)
        e_T, _ = torch.max(torch.abs(e), dim=2) # Max pooling sepanjang frekuensi
        e_T = e_T.transpose(1, 2)
        
        gat_T = self.aasist.GAT_layer_T(e_T)
        out_T = self.aasist.pool_T(gat_T)
        
        # 5. Learnable Master Node
        master1 = self.aasist.master1.expand(x.size(0), -1, -1)
        master2 = self.aasist.master2.expand(x.size(0), -1, -1)
        
        # Inference GAT 1
        out_T1, out_S1, master1 = self.aasist.HtrgGAT_layer_ST11(
            out_T, out_S, master=self.aasist.master1
        )
        out_S1 = self.aasist.pool_hS1(out_S1)
        out_T1 = self.aasist.pool_hT1(out_T1)
        
        out_T_aug, out_S_aug, master_aug = self.aasist.HtrgGAT_layer_ST12(
            out_T1, out_S1, master=master1
        )
        out_T1 = out_T1 + out_T_aug
        out_S1 = out_S1 + out_S_aug
        master1 = master1 + master_aug
        
        # Inference GAT 2
        out_T2, out_S2, master2 = self.aasist.HtrgGAT_layer_ST21(
            out_T, out_S, master=self.aasist.master2
        )
        out_S2 = self.aasist.pool_hS2(out_S2)
        out_T2 = self.aasist.pool_hT2(out_T2)
        
        out_T_aug, out_S_aug, master_aug = self.aasist.HtrgGAT_layer_ST22(
            out_T2, out_S2, master=master2
        )
        out_T2 = out_T2 + out_T_aug
        out_S2 = out_S2 + out_S_aug
        master2 = master2 + master_aug
        
        # Dropout & Feature Max/Mean
        out_T1 = self.aasist.drop_way(out_T1)
        out_T2 = self.aasist.drop_way(out_T2)
        out_S1 = self.aasist.drop_way(out_S1)
        out_S2 = self.aasist.drop_way(out_S2)
        master1 = self.aasist.drop_way(master1)
        master2 = self.aasist.drop_way(master2)
        
        out_T = torch.max(out_T1, out_T2)
        out_S = torch.max(out_S1, out_S2)
        master = torch.max(master1, master2)
        
        # Inovasi: Pooling Temporal menggunakan Temporal Attention
        attn_T = self.temporal_attention(out_T) # Shape: (batch_size, 32)
        
        # Pooling Spasial/Spectral (standar)
        S_max, _ = torch.max(torch.abs(out_S), dim=1) # Shape: (batch_size, 32)
        S_avg = torch.mean(out_S, dim=1)              # Shape: (batch_size, 32)
        
        # Konkatenasi
        last_hidden = torch.cat([attn_T, S_max, S_avg, master.squeeze(1)], dim=1) # Shape: (batch_size, 128)
        last_hidden = self.aasist.drop(last_hidden)
        
        # Output logits
        output = self.out_layer(last_hidden)
        return last_hidden, output

# Inisialisasi Proposed Model
aasist_config = {
    "first_conv": 128,
    "filts": [70, [1, 32], [32, 32], [32, 64], [64, 64]],
    "gat_dims": [64, 32],
    "pool_ratios": [0.5, 0.7, 0.5, 0.5],
    "temperatures": [2.0, 2.0, 100.0, 100.0]
}

try:
    from models.AASIST import Model as AASISTModel
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    base_model = AASISTModel(aasist_config)
    model = Proposed_AASIST_Model(base_model, fusion_dim=100, embed_dim=32).to(device)
    print(f"Proposed Model (Metode 5) berhasil diinisialisasi pada: {device}")
except ImportError as e:
    print(f"Gagal memuat arsitektur AASIST: {e}")

## 6. Tempat Import Dataset & Dataloader (ASVspoof 2019 LA)
Mendefinisikan DataLoader untuk memuat data audio waveform mentah dari ASVspoof 2019 LA.

In [ ]:
# =====================================================================
# TEMPAT UNTUK IMPORT DATASET & INISIALISASI DATALOADER (Mendukung ASVspoof 5 & 2019)
# =====================================================================
from torch.utils.data import Dataset, DataLoader
import torchaudio
import pandas as pd
import numpy as np
import os
import random
import torch

class RawAudioASVDataset(Dataset):
    """
    Dataset PyTorch untuk Audio Anti-Spoofing.
    Mendukung ASVspoof 2019 & ASVspoof 5 dengan pemindaian subfolder secara rekursif.
    Menerapkan Stratified Quota Sampling pada ASVspoof 5 untuk optimasi RAM.
    """
    def __init__(self, protocols_file, wav_dir, max_len=64000, transform=None):
        self.wav_dir = wav_dir
        self.max_len = max_len
        self.transform = transform
        self.file_list = []
        self.labels = []
        
        # Membaca berkas protokol
        if protocols_file and os.path.exists(protocols_file):
            print(f"Membaca protokol dari {protocols_file}...")
            try:
                # Cek baris pertama untuk mendeteksi tipe protokol
                with open(protocols_file, 'r', encoding='utf-8') as f:
                    first_line = f.readline().strip()
                n_cols = len(first_line.split())
                
                if n_cols >= 9: # ASVspoof 5 (10 kolom)
                    print("Mendeteksi format protokol ASVspoof 5. Menerapkan Stratified Quota Sampling...")
                    columns = [
                        "SPEAKER_ID", "FLAC_FILE_NAME", "SPEAKER_GENDER", "CODEC", "CODEC_Q",
                        "CODEC_SEED", "ATTACK_TAG", "ATTACK_LABEL", "KEY", "TMP"
                    ]
                    df = pd.read_csv(protocols_file, sep=r'\s+', header=None, names=columns[:n_cols])
                    
                    # Quota sampling (1000 bona fide, dan spoof disampling seimbang per jenis serangan)
                    max_bonafide = 1000
                    df_bonafide = df[df['KEY'] == 'bonafide']
                    df_spoof = df[df['KEY'] == 'spoof']
                    
                    n_bonafide = min(len(df_bonafide), max_bonafide)
                    df_bonafide_sampled = df_bonafide.sample(n=n_bonafide, random_state=42)
                    
                    df_spoof_sampled = pd.DataFrame()
                    if len(df_spoof) > 0:
                        attack_groups = df_spoof.groupby('ATTACK_LABEL')
                        n_groups = len(attack_groups)
                        samples_per_group = max_bonafide // n_groups
                        
                        for label, group in attack_groups:
                            n_take = min(len(group), samples_per_group)
                            df_grp_sampled = group.sample(n=n_take, random_state=42)
                            df_spoof_sampled = pd.concat([df_spoof_sampled, df_grp_sampled])
                            
                        # Penuhi sisa kuota jika ada group yang kurang sampelnya
                        current_collected = len(df_spoof_sampled)
                        if current_collected < max_bonafide:
                            needed = max_bonafide - current_collected
                            remaining_spoof = df_spoof[~df_spoof.index.isin(df_spoof_sampled.index)]
                            if len(remaining_spoof) > 0:
                                n_take = min(len(remaining_spoof), needed)
                                extra_samples = remaining_spoof.sample(n=n_take, random_state=42)
                                df_spoof_sampled = pd.concat([df_spoof_sampled, extra_samples])
                                
                    df_final = pd.concat([df_bonafide_sampled, df_spoof_sampled])
                    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
                    
                    self.file_list = df_final['FLAC_FILE_NAME'].tolist()
                    self.labels = [1 if k == 'bonafide' else 0 for k in df_final['KEY']]
                    print(f"Berhasil memuat {len(self.file_list)} sampel terstratifikasi ASVspoof 5.")
                else: # ASVspoof 2019 (5 kolom)
                    with open(protocols_file, 'r', encoding='utf-8') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) >= 5:
                                file_name = parts[1]
                                key = parts[4]
                                label = 1 if key == 'bonafide' else 0
                                self.file_list.append(file_name)
                                self.labels.append(label)
                    print(f"Berhasil memuat {len(self.file_list)} entri dataset ASVspoof 2019.")
            except Exception as e:
                print(f"Error membaca berkas protokol: {e}")
                self.file_list = []
                self.labels = []
        else:
            print(f"PERINGATAN: Berkas protokol tidak ditemukan di '{protocols_file}'.")
            print("Membuat DUMMY dataset untuk pengujian...")
            self.file_list = [f"dummy_sample_{i}" for i in range(100)]
            self.labels = [random.choice([0, 1]) for _ in range(100)]

        # Bangun indeks path file audio secara rekursif untuk mendukung folder bersarang (flac_T_aa, dll.)
        self.path_map = {}
        if os.path.exists(wav_dir):
            print(f"Membangun indeks path audio secara rekursif di {wav_dir}...")
            for root, _, files in os.walk(wav_dir):
                for f in files:
                    if f.endswith(('.flac', '.wav')):
                        name_without_ext = os.path.splitext(f)[0]
                        self.path_map[name_without_ext] = os.path.join(root, f)
            print(f"Selesai! Berhasil mengindeks {len(self.path_map)} file audio.")
        else:
            print(f"PERINGATAN: Direktori audio '{wav_dir}' tidak ditemukan.")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        label = self.labels[idx]
        
        # Cari path dari indeks map
        wav_path = self.path_map.get(file_name)
        if not wav_path:
            wav_path = os.path.join(self.wav_dir, f"{file_name}.flac")
            if not os.path.exists(wav_path):
                wav_path = os.path.join(self.wav_dir, f"{file_name}.wav")
                
        if wav_path and os.path.exists(wav_path):
            waveform, sr = torchaudio.load(wav_path)
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                waveform = resampler(waveform)
            waveform = waveform.squeeze(0)
        else:
            waveform = torch.randn(self.max_len) * 0.01
            
        # Padding atau potong agar ukuran waveform seragam
        if waveform.shape[0] < self.max_len:
            pad_len = self.max_len - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
        else:
            waveform = waveform[:self.max_len]
            
        # Terapkan transformasi (ekstraksi fitur) jika disediakan
        if self.transform:
            feature = self.transform(waveform)
            return feature, label
            
        return torch.FloatTensor(waveform), label

# SILAKAN EDIT PATH BASE DIREKTORI DI BAWAH INI SESUAI DENGAN LOKASI DATASET ASVSPOOF 5 LOKAL ANDA
BASE_DIR = "D:/Lomba/Gemastik 2026/Data Mining/panduan/Dataset_ASVspoof5_Sampled"
PATH_PROTOKOL_TRAIN = os.path.join(BASE_DIR, "ASVspoof5.train.tsv")
PATH_WAV_DIR_TRAIN = BASE_DIR

PATH_PROTOKOL_DEV = os.path.join(BASE_DIR, "ASVspoof5.dev.track_1.tsv")
PATH_WAV_DIR_DEV = BASE_DIR

print("Dataset loader ASVspoof 5 lokal siap digunakan.")


## 7. Cross-Dataset Evaluation (ASVspoof 5) & Stratified Sampling
Mendownload subset dataset streaming **ASVspoof 5** menggunakan stratified sampling untuk stress-testing model.

In [ ]:
try:
    import datasets
except ImportError:
    print("Menginstall pustaka datasets...")
    subprocess.run(["pip", "install", "datasets", "-q"])
    import datasets

from datasets import load_dataset
import pandas as pd
import numpy as np

def collect_stratified_asvspoof5_tsv(tsv_path, max_samples=5000):
    """
    Membaca berkas protokol TSV ASVspoof 5 secara lokal,
    melakukan stratified sampling:
    - Bona fide: 50%, Spoof: 50%
    - Spoof tersampel secara merata berdasarkan Attack ID (ATTACK_LABEL)
    """
    columns = [
        "SPEAKER_ID", "FLAC_FILE_NAME", "SPEAKER_GENDER", "CODEC", "CODEC_Q",
        "CODEC_SEED", "ATTACK_TAG", "ATTACK_LABEL", "KEY", "TMP"
    ]
    print(f"Membaca berkas protokol dari {tsv_path}...")
    df = pd.read_csv(tsv_path, sep=r'\s+', header=None, names=columns)
    print(f"Total baris ditemukan: {len(df)}")
    
    target_class_samples = max_samples // 2
    
    # Pisahkan bona fide dan spoof
    df_bonafide = df[df['KEY'] == 'bonafide']
    df_spoof = df[df['KEY'] == 'spoof']
    
    # Ambil sampel bona fide
    n_bonafide = min(len(df_bonafide), target_class_samples)
    df_bonafide_sampled = df_bonafide.sample(n=n_bonafide, random_state=42)
    
    # Stratifikasi sampel spoof berdasarkan ATTACK_LABEL
    df_spoof_sampled = pd.DataFrame()
    if len(df_spoof) > 0:
        attack_groups = df_spoof.groupby('ATTACK_LABEL')
        n_groups = len(attack_groups)
        samples_per_group = target_class_samples // n_groups
        
        # Ambil sampel dari setiap group secara merata
        for label, group in attack_groups:
            n_take = min(len(group), samples_per_group)
            df_grp_sampled = group.sample(n=n_take, random_state=42)
            df_spoof_sampled = pd.concat([df_spoof_sampled, df_grp_sampled])
            
        current_collected = len(df_spoof_sampled)
        if current_collected < target_class_samples:
            needed = target_class_samples - current_collected
            remaining_spoof = df_spoof[~df_spoof.index.isin(df_spoof_sampled.index)]
            if len(remaining_spoof) > 0:
                n_take = min(len(remaining_spoof), needed)
                extra_samples = remaining_spoof.sample(n=n_take, random_state=42)
                df_spoof_sampled = pd.concat([df_spoof_sampled, extra_samples])
                
    # Gabungkan
    df_final = pd.concat([df_bonafide_sampled, df_spoof_sampled])
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    
    samples_list = []
    for _, row in df_final.iterrows():
        samples_list.append({
            'file_name': row['FLAC_FILE_NAME'],
            'label': 1 if row['KEY'] == 'bonafide' else 0,
            'system_id': row['ATTACK_LABEL']
        })
        
    print(f"Selesai! Mengumpulkan {len(df_bonafide_sampled)} Bona fide dan {len(df_spoof_sampled)} Spoof.")
    return samples_list

def collect_stratified_asvspoof5(dataset_stream, max_samples=5000):
    """
    Mengambil sampel terstratifikasi secara round-robin dari dataset streaming ASVspoof 5 (Hugging Face fallback).
    """
    target_class_samples = max_samples // 2
    bona_fide_samples = []
    spoof_by_attack = {}
    
    print("Mengekstrak sampel dari streaming dataset... Mohon tunggu sebentar.")
    
    for item in dataset_stream:
        label = item.get('label')
        if label is None:
            label = item.get('key') or item.get('class')
            
        is_bonafide = False
        if label in [1, '1', 'bonafide', 'bona_fide', 'human']:
            is_bonafide = True
        elif label in [0, '0', 'spoof', 'spoofed', 'attack']:
            is_bonafide = False
        else:
            is_bonafide = 'bonafide' in str(label).lower() or 'human' in str(label).lower()
            
        if is_bonafide:
            if len(bona_fide_samples) < target_class_samples:
                bona_fide_samples.append(item)
        else:
            attack_id = item.get('attack_type') or item.get('attack_id') or item.get('system_id') or item.get('attack') or 'unknown'
            if attack_id not in spoof_by_attack:
                spoof_by_attack[attack_id] = []
            spoof_by_attack[attack_id].append(item)
            
        total_spoof_collected = sum(len(v) for v in spoof_by_attack.values())
        if len(bona_fide_samples) >= target_class_samples and total_spoof_collected >= target_class_samples * 2:
            break
            
    spoof_samples = []
    if spoof_by_attack:
        attack_ids = list(spoof_by_attack.keys())
        idx = 0
        while len(spoof_samples) < target_class_samples:
            added = False
            for aid in attack_ids:
                if idx < len(spoof_by_attack[aid]):
                    spoof_samples.append(spoof_by_attack[aid][idx])
                    added = True
                    if len(spoof_samples) >= target_class_samples:
                        break
            if not added:
                break
            idx += 1
            
    final_samples = bona_fide_samples + spoof_samples
    random.shuffle(final_samples)
    print(f"Selesai! Mengumpulkan {len(bona_fide_samples)} Bona fide dan {len(spoof_samples)} Spoof.")
    return final_samples

# Jalur lokal untuk berkas manifes dan direktori audio
PATH_ASV5_TSV = "D:/Lomba/Gemastik 2026/Data Mining/panduan/Dataset_ASVspoof5_Sampled/ASVspoof5.dev.track_1.tsv"
if not os.path.exists(PATH_ASV5_TSV):
    PATH_ASV5_TSV = "../../Dataset_ASVspoof5_Sampled/ASVspoof5.dev.track_1.tsv"
    
PATH_ASV5_WAV_DIR = "D:/Lomba/Gemastik 2026/Data Mining/panduan/dataset/asvspoof5/dev/flac"

try:
    if os.path.exists(PATH_ASV5_TSV):
        print(f"Memuat dataset dari berkas TSV lokal: {PATH_ASV5_TSV}")
        eval_samples = collect_stratified_asvspoof5_tsv(PATH_ASV5_TSV, max_samples=5000)
        use_local_asv5 = True
    else:
        print("Berkas TSV lokal tidak ditemukan. Menggunakan streaming dari Hugging Face...")
        asv5_dataset = load_dataset("jungjee/asvspoof5", streaming=True)
        stream_split = asv5_dataset['train']
        eval_samples = collect_stratified_asvspoof5(stream_split, max_samples=5000)
        use_local_asv5 = False
except Exception as e:
    print(f"Error memuat dataset ASVspoof 5: {e}")
    raise e


## 8. Metrik Evaluasi: Equal Error Rate (EER) & min t-DCF
Kita mendefinisikan fungsi perhitungan EER dan min t-DCF.

In [ ]:
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def compute_eer_scipy(bonafide_scores, spoof_scores):
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    y_score = np.concatenate([bonafide_scores, spoof_scores])
    
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    
    eer = brentq(lambda x: interp1d(fpr, tpr)(x) + x - 1., 0., 1.)
    return eer

def compute_asvspoof19_min_tdcf(bonafide_scores, spoof_scores, Pfa_asv=0.22276, Pmiss_asv=0.06677, Pmiss_spoof_asv=0.76843):
    Pspoof = 0.05
    Ptar = (1.0 - Pspoof) * 0.99
    Pnon = (1.0 - Pspoof) * 0.01
    
    Cmiss_cm = 1.0
    Cfa_cm = 10.0
    Cmiss_asv = 1.0
    Cfa_asv = 10.0
    
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    y_score = np.concatenate([bonafide_scores, spoof_scores])
    
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    Pmiss_cm = 1.0 - tpr
    Pfa_cm = fpr
    
    C1 = Ptar * (Cmiss_cm - Cmiss_asv * Pmiss_asv) - Pnon * Cfa_asv * Pfa_asv
    C2 = Cfa_cm * Pspoof * (1.0 - Pmiss_spoof_asv)
    
    tDCF = C1 * Pmiss_cm + C2 * Pfa_cm
    tDCF_norm = tDCF / np.minimum(C1, C2)
    min_tdcf = np.min(tDCF_norm)
    return min_tdcf

print("Fungsi evaluasi metrik siap.")

## 9. Training & Validation Loop
Model Proposed dilatih dengan optimizer `AdamW` dan scheduler `CosineAnnealingLR`. Fitur fusi diekstraksi secara on-the-fly, model Proposed dilatih dan divalidasi, serta dievaluasi pada dataset ASVspoof 5.

In [ ]:
import torch.optim as optim

def train_one_epoch(model, dataloader, optimizer, criterion, device, feature_extractor):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        # Ekstraksi fitur hibrida CQT-LFCC secara on-the-fly
        with torch.no_grad():
            features = feature_extractor(batch_x)
            
        optimizer.zero_grad()
        _, outputs = model(features)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

def validate_model(model, dataloader, criterion, device, feature_extractor):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    bonafide_scores = []
    spoof_scores = []
    
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            features = feature_extractor(batch_x)
            
            _, outputs = model(features)
            loss = criterion(outputs, batch_y)
            
            running_loss += loss.item() * batch_x.size(0)
            _, predicted = outputs.max(1)
            total += batch_y.size(0)
            correct += predicted.eq(batch_y).sum().item()
            
            probs = F.softmax(outputs, dim=1)
            bona_probs = probs[:, 1].cpu().numpy()
            labels_np = batch_y.cpu().numpy()
            
            for score, lbl in zip(bona_probs, labels_np):
                if lbl == 1:
                    bonafide_scores.append(score)
                else:
                    spoof_scores.append(score)
                    
    val_loss = running_loss / total
    val_acc = (correct / total) * 100
    
    bonafide_scores = np.array(bonafide_scores)
    spoof_scores = np.array(spoof_scores)
    
    if len(bonafide_scores) > 0 and len(spoof_scores) > 0:
        val_eer = compute_eer_scipy(bonafide_scores, spoof_scores) * 100
        val_tdcf = compute_asvspoof19_min_tdcf(bonafide_scores, spoof_scores)
    else:
        val_eer = 100.0
        val_tdcf = 1.0
        
    return val_loss, val_acc, val_eer, val_tdcf

print("Fungsi Train & Validate siap digunakan.")

In [ ]:
# Parameter Pelatihan
num_epochs = 3  # Sesuaikan untuk eksperimen nyata
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

# Inisialisasi Ekstraktor Fitur Hibrida
feature_extractor = CQTLFCCFeatureExtractor(sample_rate=16000, n_bins_cqt=80, n_lfcc=20, target_time_len=2000).to(device)

# Pemuatan Dataset
train_dataset = RawAudioASVDataset(protocols_file=PATH_PROTOKOL_TRAIN, wav_dir=PATH_WAV_DIR_TRAIN, max_len=64000)
dev_dataset = RawAudioASVDataset(protocols_file=PATH_PROTOKOL_DEV, wav_dir=PATH_WAV_DIR_DEV, max_len=64000)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=8, shuffle=False)

# Loop Pelatihan
print(f"\nMemulai Pelatihan Proposed Model CQT-LFCC + AASIST + Temporal Attention selama {num_epochs} Epoch...\n")
print(f"{'Epoch':^8} | {'Train Loss':^12} | {'Train Acc (%)':^15} | {'Val Loss':^12} | {'Val Acc (%)':^15} | {'Val EER (%)':^15} | {'min t-DCF':^12}")
print("-" * 105)

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        feature_extractor=feature_extractor
    )
    
    val_loss, val_acc, val_eer, val_tdcf = validate_model(
        model=model,
        dataloader=dev_loader,
        criterion=criterion,
        device=device,
        feature_extractor=feature_extractor
    )
    
    scheduler.step()
    
    print(f"{epoch:^8} | {train_loss:^12.4f} | {train_acc:^15.2f} | {val_loss:^12.4f} | {val_acc:^15.2f} | {val_eer:^15.4f} | {val_tdcf:^12.5f}")

print("\nPelatihan selesai!")

class ASVspoof5EvalDataset(Dataset):
    """
    Dataset PyTorch untuk memuat data evaluasi ASVspoof 5 (mendukung mode lokal & streaming HF).
    Mendukung struktur folder bersarang dengan pencarian path secara rekursif.
    """
    def __init__(self, samples, wav_dir, max_len=64000, use_local=True):
        self.samples = samples
        self.wav_dir = wav_dir
        self.max_len = max_len
        self.use_local = use_local
        
        # Bangun indeks path file audio secara rekursif jika menggunakan mode lokal
        self.path_map = {}
        if self.use_local and os.path.exists(wav_dir):
            print(f"Membangun indeks path audio secara rekursif di {wav_dir}...")
            for root, _, files in os.walk(wav_dir):
                for f in files:
                    if f.endswith(('.flac', '.wav')):
                        name_without_ext = os.path.splitext(f)[0]
                        self.path_map[name_without_ext] = os.path.join(root, f)
            print(f"Selesai! Berhasil mengindeks {len(self.path_map)} file audio.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        
        if self.use_local:
            file_name = item['file_name']
            label = item['label']
            
            # Cari path dari indeks map
            wav_path = self.path_map.get(file_name)
            
            # Jika tidak ditemukan di map, fallback ke os.path.join standar
            if not wav_path:
                wav_path = os.path.join(self.wav_dir, f"{file_name}.flac")
                if not os.path.exists(wav_path):
                    wav_path = os.path.join(self.wav_dir, f"{file_name}.wav")
                
            if wav_path and os.path.exists(wav_path):
                waveform, sr = torchaudio.load(wav_path)
                if sr != 16000:
                    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                    waveform = resampler(waveform)
                waveform = waveform.squeeze(0)
            else:
                waveform = torch.randn(self.max_len) * 0.01
        else:
            audio_data = item['audio']['array']
            sr = item['audio']['sampling_rate']
            waveform = torch.tensor(audio_data, dtype=torch.float32)
            
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                waveform = resampler(waveform)
                
            if waveform.ndim > 1:
                waveform = waveform.mean(dim=0)
                
            label = item.get('label')
            if label is None:
                label = item.get('key') or item.get('class')
                
            is_bonafide = False
            if label in [1, '1', 'bonafide', 'bona_fide', 'human']:
                is_bonafide = True
            elif label in [0, '0', 'spoof', 'spoofed', 'attack']:
                is_bonafide = False
            else:
                is_bonafide = 'bonafide' in str(label).lower() or 'human' in str(label).lower()
            label = 1 if is_bonafide else 0
            
        if waveform.shape[0] < self.max_len:
            pad_len = self.max_len - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
        else:
            waveform = waveform[:self.max_len]
            
        return waveform, label
